In [1]:
print("Hello World")

Hello World


In [2]:
!pip install transformers datasets peft accelerate evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.9 MB/s eta 0:00:00:00:010:01
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=88e3508a2f6edbf54efe6900219f039b3132f15cb59609acc7d2478a8ae19fa0
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1

In [3]:
import json 

data_path = "/kaggle/input/datasets/alwinchetty/skinny/synthetic_ner_65537.json"

with open(data_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

In [4]:
# extract the dataset list

dataset_list = raw_data['dataset']

In [5]:
print(f"Total samples loaded: {len(dataset_list)}")
print("\nSample 0 text:")
print(dataset_list[0]["text"])
print("\nSample 0 entities:")
print(json.dumps(dataset_list[0]["entities"], indent=2))


Total samples loaded: 65537

Sample 0 text:
Required: Bachelor's degree in Software Engineering. Junior Print Designer at Revolut, Montreal. onsite. Skills: Sinatra, Creo.

Sample 0 entities:
[
  {
    "start": 10,
    "end": 51,
    "label": "EDUCATION",
    "text": "Bachelor's degree in Software Engineering"
  },
  {
    "start": 53,
    "end": 59,
    "label": "SENIORITY",
    "text": "Junior"
  },
  {
    "start": 60,
    "end": 74,
    "label": "ROLE",
    "text": "Print Designer"
  },
  {
    "start": 78,
    "end": 85,
    "label": "ORGANIZATION",
    "text": "Revolut"
  },
  {
    "start": 87,
    "end": 95,
    "label": "LOCATION",
    "text": "Montreal"
  },
  {
    "start": 97,
    "end": 103,
    "label": "WORK_MODE",
    "text": "onsite"
  },
  {
    "start": 113,
    "end": 120,
    "label": "TECHNOLOGY",
    "text": "Sinatra"
  },
  {
    "start": 122,
    "end": 126,
    "label": "TECHNOLOGY",
    "text": "Creo"
  }
]


In [6]:
base_entities = [
    "ROLE", 
    "TECHNOLOGY", 
    "ORGANIZATION", 
    "EXPERIENCE", 
    "SENIORITY", 
    "EDUCATION", 
    "LOCATION", 
    "WORK_MODE"
]

In [7]:
label_list = ['0']
for entity in base_entities:
    label_list.append(f"B-{entity}")
    label_list.append(f"I-{entity}")    

In [8]:
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# label2id = {}

# for label, id in enumerate(label_list):
#     label2id[label] = id

In [9]:
label2id

{'0': 0,
 'B-ROLE': 1,
 'I-ROLE': 2,
 'B-TECHNOLOGY': 3,
 'I-TECHNOLOGY': 4,
 'B-ORGANIZATION': 5,
 'I-ORGANIZATION': 6,
 'B-EXPERIENCE': 7,
 'I-EXPERIENCE': 8,
 'B-SENIORITY': 9,
 'I-SENIORITY': 10,
 'B-EDUCATION': 11,
 'I-EDUCATION': 12,
 'B-LOCATION': 13,
 'I-LOCATION': 14,
 'B-WORK_MODE': 15,
 'I-WORK_MODE': 16}

In [10]:
id2label

{0: '0',
 1: 'B-ROLE',
 2: 'I-ROLE',
 3: 'B-TECHNOLOGY',
 4: 'I-TECHNOLOGY',
 5: 'B-ORGANIZATION',
 6: 'I-ORGANIZATION',
 7: 'B-EXPERIENCE',
 8: 'I-EXPERIENCE',
 9: 'B-SENIORITY',
 10: 'I-SENIORITY',
 11: 'B-EDUCATION',
 12: 'I-EDUCATION',
 13: 'B-LOCATION',
 14: 'I-LOCATION',
 15: 'B-WORK_MODE',
 16: 'I-WORK_MODE'}

In [11]:
from datasets import load_dataset

from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer, 
    DataCollatorForTokenClassification
)

from peft import (
    LoraConfig,
    get_peft_model, 
    TaskType
)

import torch

In [12]:
MODEL_NAME = "microsoft/deberta-v3-base"

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels = len(label_list),
    id2label = id2label,
    label2id = label2id
)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; 

In [13]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# tokenizer.pad_token = tokenizer.eos_token 
# this padding is only used for text generation models or llms because they dont have padding built-in
# since deberta is a encoder model it has built-in padding. 

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [14]:
dataset_list[-1]

{'text': 'We need an exceptional Mid-Level Content Manager to join Figma in Santa Monica. The candidate should have 5 years of experience in Capture One. hybrid arrangement available.',
 'entities': [{'start': 23,
   'end': 32,
   'label': 'SENIORITY',
   'text': 'Mid-Level'},
  {'start': 33, 'end': 48, 'label': 'ROLE', 'text': 'Content Manager'},
  {'start': 57, 'end': 62, 'label': 'ORGANIZATION', 'text': 'Figma'},
  {'start': 66, 'end': 78, 'label': 'LOCATION', 'text': 'Santa Monica'},
  {'start': 106, 'end': 113, 'label': 'EXPERIENCE', 'text': '5 years'},
  {'start': 131, 'end': 142, 'label': 'TECHNOLOGY', 'text': 'Capture One'},
  {'start': 144, 'end': 150, 'label': 'WORK_MODE', 'text': 'hybrid'}]}

In [15]:
formatted_dataset = []

for i in dataset_list:
    # 1. Tokenize the text and get the offsets
    tokenized_input = tokenizer(
        i['text'],
        return_offsets_mapping=True,
        truncation=True
    )
    
    input_ids = tokenized_input['input_ids']
    offset_mapping = tokenized_input['offset_mapping']
    
    # 2. Initialize labels: -100 for special tokens (like [CLS], [SEP]), 0 for word tokens
    sequence_ids = tokenized_input.sequence_ids()
    labels = [-100 if seq_id is None else 0 for seq_id in sequence_ids]
    
    # 3. Align the character offsets to the tokens
    for entity in i['entities']:
        start_char = entity['start']
        end_char = entity['end']
        label_name = entity['label']
        
        # Get BIO label IDs
        b_label = label2id[f"B-{label_name}"]
        i_label = label2id[f"I-{label_name}"]
        
        is_first = True
        for idx, (start_tok, end_tok) in enumerate(offset_mapping):
            # Skip special tokens
            if labels[idx] == -100:
                continue
                
            # Check if token overlaps with the entity character span
            if start_tok < end_char and end_tok > start_char:
                if is_first:
                    labels[idx] = b_label
                    is_first = False
                else:
                    labels[idx] = i_label
                    
    # 4. Save the processed sample
    formatted_dataset.append({
        'input_ids': input_ids,
        'attention_mask': tokenized_input['attention_mask'],
        'labels': labels
    })

print(f"Successfully formatted {len(formatted_dataset)} samples!")


Successfully formatted 65537 samples!


In [16]:
from datasets import Dataset

formatted_dataset = Dataset.from_list(formatted_dataset)

In [17]:
formatted_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 65537
})

In [18]:
train_test_split = formatted_dataset.train_test_split(
    0.1, seed = 42
)

In [19]:
lora_config = LoraConfig(
    task_type = TaskType.TOKEN_CLS,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    target_modules = [
        'query_proj',
        'value_proj'
    ],
    bias = 'none'
)

In [20]:
model = get_peft_model(
    model,
    lora_config
)

In [21]:
model.print_trainable_parameters()

trainable params: 602,897 || all params: 184,447,522 || trainable%: 0.3269


In [22]:
data_collator = DataCollatorForTokenClassification(
    tokenizer = tokenizer
)

In [23]:
import evaluate
import numpy as np

# Load the seqeval metric
metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    
    # Step 5: Get the predicted class index (highest probability) for each token
    predictions = np.argmax(predictions, axis=-1)
    
    # Step 6: Clean predictions and labels (filter out -100 and convert IDs to string labels)
    true_predictions = []
    true_labels = []
    
    for prediction, label in zip(predictions, labels):
        true_pred = []
        true_lab = []
        for p, l in zip(prediction, label):
            if l != -100:  # Filter out special tokens and padding
                true_pred.append(id2label[p])
                true_lab.append(id2label[l])
        true_predictions.append(true_pred)
        true_labels.append(true_lab)
        
    # Step 7: Compute overall precision, recall, f1, and accuracy
    results = metric.compute(predictions=true_predictions, references=true_labels)
    
    # Return a flat dictionary containing only the main metrics we care about
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [24]:
training_args = TrainingArguments(
    output_dir = './results',
    learning_rate = 1e-4,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.01,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model = 'f1',
    # fp16 = True, kaggle gpu doesnt support this or fp32
    logging_steps = 100,
    warmup_steps = 1100,
    lr_scheduler_type = 'linear'
)

In [25]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_test_split['train'],
    eval_dataset = train_test_split['test'],
    processing_class = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

In [26]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.676962,0.077515,0.989086,0.991445,0.990264,0.995837
2,0.125944,0.052612,0.994165,0.996008,0.995085,0.997927
3,0.053678,0.030609,0.998224,0.998030,0.998127,0.998864


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not

TrainOutput(global_step=5532, training_loss=0.5751866360376128, metrics={'train_runtime': 1324.2171, 'train_samples_per_second': 133.625, 'train_steps_per_second': 4.178, 'total_flos': 3775564070044284.0, 'train_loss': 0.5751866360376128, 'epoch': 3.0})

In [27]:
!zip -r trained_ner_model.zip /kaggle/working/results/checkpoint-5532

  adding: kaggle/working/results/checkpoint-5532/ (stored 0%)
  adding: kaggle/working/results/checkpoint-5532/trainer_state.json (deflated 74%)
  adding: kaggle/working/results/checkpoint-5532/tokenizer_config.json (deflated 48%)
  adding: kaggle/working/results/checkpoint-5532/optimizer.pt (deflated 9%)
  adding: kaggle/working/results/checkpoint-5532/scheduler.pt (deflated 61%)
  adding: kaggle/working/results/checkpoint-5532/adapter_config.json (deflated 56%)
  adding: kaggle/working/results/checkpoint-5532/README.md (deflated 66%)
  adding: kaggle/working/results/checkpoint-5532/training_args.bin (deflated 53%)
  adding: kaggle/working/results/checkpoint-5532/rng_state.pth (deflated 26%)
  adding: kaggle/working/results/checkpoint-5532/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/results/checkpoint-5532/tokenizer.json (deflated 79%)
